# FINAL PROJECT
## Group 2
WQD7013 Statistics for Data Science
* Dataset E – WHO + World Bank Merged Health & Development Panel (Global Health)


## Part 1 - Dataset Audit and Exploratory Analysis


---



### 1.1 Structural Audit and Variable Classification

In [4]:
import pandas as pd

df = pd.read_csv("who_wb_merged_2000_2022.csv")

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'who_wb_merged_2000_2022.csv'

#### 1.1(a) Variable Inventory

This section provides an overview of all variables in the dataset, including their inferred data type, corrected data type, measurement level, and analytical role. This classification is essential to ensure appropriate statistical methods are applied in subsequent analysis.

Variables are classified into identifiers, predictors, outcomes, and control variables based on the primary research objective, which focuses on understanding factors influencing health outcomes such as under-five mortality rates.

In [ ]:
# Create variable inventory
inventory = pd.DataFrame({
    "Variable": df.columns,
    "Inferred_dtype": df.dtypes.astype(str)
})

inventory

In [ ]:
# Add additional columns to complete inventory

inventory["Correct_Type"] = [
    "category","category","int","category","category","category",
    "float","float","float","float",
    "float","float","float","float",
    "float","float","float","float","float",
    "float","float","float",
    "float","float","float","float"
]

inventory["Measurement_Level"] = [
    "Nominal","Nominal","Discrete","Ordinal","Nominal","Nominal",
    "Continuous","Continuous","Continuous","Continuous",
    "Continuous","Continuous","Continuous","Continuous",
    "Continuous","Continuous","Continuous","Continuous","Continuous",
    "Continuous","Continuous","Continuous",
    "Continuous","Continuous","Continuous","Continuous"
]

inventory["Role"] = [
    "Identifier","Identifier","Identifier",
    "Predictor","Control","Control",
    "Control","Predictor","Predictor","Predictor",
    "Predictor","Predictor","Predictor","Predictor",
    "Outcome","Outcome","Outcome","Outcome","Outcome",
    "Predictor","Predictor","Predictor",
    "Outcome","Predictor","Predictor","Derived"
]

inventory

In [ ]:
inventory.to_csv("variable_inventory.csv", index=False)

In [ ]:
# Display full table nicely
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

inventory


In [ ]:
pd.set_option('display.max_columns', None)
inventory


#### 1.1(b) Ambiguous Measurement Level

The variable *wb_income_group* presents ambiguity in its measurement level classification. Although it is stored as a categorical variable with labels such as low income, lower-middle income, upper-middle income, and high income, it inherently reflects an ordered hierarchy based on economic development. In this dataset, higher income groups are generally associated with higher GDP per capita and better health outcomes, including lower under-five mortality rates (u5mr). Therefore, it is classified as an ordinal variable.

If treated as a nominal variable, the analysis would ignore the natural ordering and require separate dummy variables, reducing interpretability. Conversely, treating it as continuous would incorrectly assume equal spacing between categories, which is unrealistic. Such misclassification could lead to biased estimates and misleading conclusions in regression analysis.

In [ ]:
df = pd.read_csv('who_wb_merged_2000_2022.csv')


In [ ]:
df.head()
df.info()


In [ ]:
# Calculate missing values
missing_count = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

# Create table
missing_table = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %': missing_percent
}).sort_values(by='Missing %', ascending=False)

missing_table

In [ ]:
missing_table.head(10)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
missing_percent.sort_values(ascending=False).plot(kind='bar')
plt.title('Missing Data Percentage by Variable')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=45)
plt.show()


The bar chart clearly shows:

Several variables reach 100% missingness, indicating they are unusable for analysis
A sharp drop in missingness for other variables
Missing values are not evenly distributed across variables

In [ ]:
import seaborn as sns

plt.figure(figsize=(12,6))
sns.heatmap(df.isnull(), cbar=False)
plt.title('Missing Data Heatmap')
plt.show()

In [ ]:
plt.figure(figsize=(12,6))
sns.heatmap(df.sample(300).isnull(), cbar=False)
plt.title('Missing Data Heatmap (Sampled)')
plt.show()

In [ ]:
import missingno as msno

msno.matrix(df)

In [ ]:
df_clean = df.drop(columns=[
    'life_expectancy_male',
    'life_expectancy_female',
    'life_expectancy_both',
    'hiv_prevalence_pct',
    'life_expectancy_gap_f_minus_m'
])

In [ ]:
df_clean['health_exp_per_capita_usd'] = df_clean.groupby('wb_income_group')['health_exp_per_capita_usd'].transform(lambda x: x.fillna(x.median()))

df_clean['maternal_mortality_ratio'] = df_clean.groupby('wb_income_group')['maternal_mortality_ratio'].transform(lambda x: x.fillna(x.median()))

In [ ]:
df_clean['sanitation_pct'].fillna(df_clean['sanitation_pct'].median(), inplace=True)

In [ ]:
df_clean.columns

In [ ]:
df_clean.isnull().sum()

In [ ]:
df_clean = df.copy()

In [ ]:
drop_cols = [
    'life_expectancy_male',
    'life_expectancy_female',
    'life_expectancy_both',
    'hiv_prevalence_pct',
    'life_expectancy_gap_f_minus_m'
]

df_clean = df_clean.drop(columns=drop_cols)


In [ ]:
high_missing_cols = ['smoking_pct', 'physicians_per_1000']
df_clean = df_clean.drop(columns=high_missing_cols)


In [ ]:
group_vars = [
    'health_exp_per_capita_usd',
    'maternal_mortality_ratio',
    'sanitation_pct',
    'dtp3_coverage_pct',
    'measles_coverage_pct',
    'log_health_exp'
]

for col in group_vars:
    df_clean[col] = df_clean.groupby('wb_income_group')[col]\
        .transform(lambda x: x.fillna(x.median()))

In [ ]:
df_clean['gdp_per_capita_ppp'] = df_clean['gdp_per_capita_ppp'].fillna(df_clean['gdp_per_capita_ppp'].median())

df_clean['u5mr'] = df_clean['u5mr'].fillna(df_clean['u5mr'].median())

df_clean['safe_water_pct'] = df_clean['safe_water_pct'].fillna(df_clean['safe_water_pct'].median())


In [ ]:
(df_clean.isnull().sum() / len(df_clean) * 100).sort_values(ascending=False)


In [ ]:
df_clean = df_clean.drop(columns=['log_u5mr', 'log_gdp_per_capita'])

In [ ]:
df_clean = df_clean.drop(columns=['obesity_pct'])


In [ ]:
df_clean['who_region'] = df_clean['who_region'].fillna(df_clean['who_region'].mode()[0])

In [ ]:
(df_clean.isnull().sum() / len(df_clean) * 100).sort_values(ascending=False)


Following further refinement, remaining derived variables and variables with moderate missingness were either removed or appropriately imputed. As a result, the final dataset contains negligible or no missing values across all retained variables, ensuring suitability for robust statistical modelling without introducing bias.

In [ ]:
print("Before cleaning:", df['health_exp_per_capita_usd'].mean())
print("After cleaning:", df_clean['health_exp_per_capita_usd'].mean())


In [ ]:
print("Std before:", df['health_exp_per_capita_usd'].std())
print("Std after:", df_clean['health_exp_per_capita_usd'].std())


### 1.2 Univariate Distributions

#### 1.2(a)

Summary Statistics of Continuous Variables
Summary statistics were computed for all continuous variables, including measures of central tendency (mean and median), dispersion (standard deviation and interquartile range), and distribution shape (skewness and excess kurtosis), along with the 5th and 95th percentiles. These metrics provide insights into variability, potential outliers, and the overall distributional characteristics of the data.
Interpretation
Several variables, including gdp_per_capita_ppp, health_exp_per_capita_usd, maternal_mortality_ratio, and u5mr, exhibit strong positive skewness, indicating the presence of extreme high values and long right tails. The population variable shows extremely high skewness and kurtosis, suggesting substantial outliers.
In contrast, variables such as safe_water_pct, sanitation_pct, and immunisation indicators display negative skewness, reflecting distributions bounded near upper limits. The transformed variable log_health_exp appears approximately symmetric, confirming that the logarithmic transformation effectively stabilised the distribution.
These findings highlight the need for transformations and careful handling of skewed variables in subsequent modelling.

In [ ]:
df_clean.columns

In [ ]:
continuous_vars = [
    'gdp_per_capita_ppp',
    'health_exp_per_capita_usd',
    'population',
    'safe_water_pct',
    'sanitation_pct',
    'dtp3_coverage_pct',
    'measles_coverage_pct',
    'u5mr',
    'maternal_mortality_ratio',
    'log_health_exp'
]

In [ ]:
import pandas as pd

summary = pd.DataFrame()

for col in continuous_vars:
    summary.loc[col, 'Mean'] = df_clean[col].mean()
    summary.loc[col, 'Median'] = df_clean[col].median()
    summary.loc[col, 'SD'] = df_clean[col].std()
    summary.loc[col, 'IQR'] = df_clean[col].quantile(0.75) - df_clean[col].quantile(0.25)
    summary.loc[col, 'Skewness'] = df_clean[col].skew()
    summary.loc[col, 'Excess Kurtosis'] = df_clean[col].kurt()
    summary.loc[col, 'P5'] = df_clean[col].quantile(0.05)
    summary.loc[col, 'P95'] = df_clean[col].quantile(0.95)

summary.round(2)

#### 1.2 (b)

In [ ]:
import matplotlib.pyplot as plt

for col in continuous_vars:
    plt.figure()
    plt.hist(df_clean[col], bins=20)
    plt.title(f'Histogram of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.show()


In [ ]:
import numpy as np

selected_vars = ['gdp_per_capita_ppp', 'health_exp_per_capita_usd', 'safe_water_pct']

for col in selected_vars:
    data = df_clean[col]

    IQR = data.quantile(0.75) - data.quantile(0.25)
    n = len(data)

    bin_width = (2 * IQR) / (n ** (1/3))
    bins = int((data.max() - data.min()) / bin_width)

    print(f"{col}:")
    print(f"  IQR = {IQR:.2f}")
    print(f"  Bin width = {bin_width:.2f}")
    print(f"  Suggested bins = {bins}")
    print("------")


In [ ]:
for col in selected_vars:
    IQR = df_clean[col].quantile(0.75) - df_clean[col].quantile(0.25)
    n = len(df_clean[col])
    bin_width = (2 * IQR) / (n ** (1/3))
    bins = int((df_clean[col].max() - df_clean[col].min()) / bin_width)

    plt.figure()
    plt.hist(df_clean[col], bins=bins)
    plt.title(f'Histogram of {col} (FD rule)')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.show()

Histograms were constructed for all continuous variables to visualise their distributions. For selected variables, the number of bins was determined using the Freedman–Diaconis rule, which adapts bin width based on the interquartile range (IQR) and sample size.
The histograms indicate that variables such as gdp_per_capita_ppp and health_exp_per_capita_usd are strongly right-skewed, while safe_water_pct shows left-skewness due to its bounded nature. These patterns are consistent with the summary statistics and highlight the need for transformation in subsequent analysis.

#### 1.2(c) Transformation and Normality Assessment
The primary outcome variable, u5mr (under-five mortality rate), was assessed for normality using a Q-Q plot and the Shapiro–Wilk test. The Q-Q plot of the original variable shows a strong deviation from the reference line, indicating substantial positive skewness and departure from normality. This is supported by the Shapiro–Wilk test, which yields a very small p-value, confirming non-normality.
To address this issue, a logarithmic transformation was applied to the variable. The Q-Q plot of log_u5mr shows a clear improvement, with observations more closely aligned along the diagonal line. Although the normality test still rejects strict normality, the distribution is considerably more symmetric after transformation.
The skewness coefficient decreased substantially after transformation, indicating reduced asymmetry. This demonstrates that the log transformation effectively stabilises the distribution, making it more suitable for subsequent statistical analysis.

Q-Q plot original

In [2]:
import scipy.stats as stats
import matplotlib.pyplot as plt

plt.figure()
stats.probplot(df_clean['u5mr'], dist="norm", plot=plt)
plt.title("Q-Q Plot of u5mr (Original)")
plt.show()

NameError: name 'df_clean' is not defined

<Figure size 640x480 with 0 Axes>

In [ ]:
stats.shapiro(df_clean['u5mr'])

In [ ]:
import numpy as np

df_clean['log_u5mr'] = np.log(df_clean['u5mr'])

Q-Q plot transform

In [ ]:
plt.figure()
stats.probplot(df_clean['log_u5mr'], dist="norm", plot=plt)
plt.title("Q-Q Plot of log_u5mr")
plt.show()

In [ ]:
stats.shapiro(df_clean['log_u5mr'])


In [ ]:
print("Skewness before:", df_clean['u5mr'].skew())
print("Skewness after:", df_clean['log_u5mr'].skew())

#### 1.2 (d) Categorical Variables

In [ ]:
categorical_vars = [
    'wb_income_group',
    'wb_region',
    'who_region',
    'year'  # treat as categorical for frequency
]


In [ ]:
for col in categorical_vars:
    print(f"\nVariable: {col}")

    freq = df_clean[col].value_counts()
    prop = df_clean[col].value_counts(normalize=True) * 100

    table = pd.DataFrame({
        'Frequency': freq,
        'Proportion (%)': prop.round(2)
    })

    print(table)
    print("\nCategories with <30 observations:")
    print(freq[freq < 30])


Frequencies and proportions were computed for all categorical variables to assess the distribution of categories. No categories with fewer than 30 observations were identified, indicating that no rare categories are present.

As a result, no collapsing or removal is required, and all categorical variables are retained. The variable year, although containing multiple categories, is treated as a temporal variable and will be incorporated appropriately in modelling.

### 1.3 Bivariate Relationships



#### 1.3 (a) Pearson correlation matrix

In [ ]:
corr_matrix = df_clean[continuous_vars].corr(method='pearson')
corr_matrix.round(2)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Pearson Correlation Heatmap")
plt.show()

In [ ]:
import numpy as np

# Remove self-correlation
corr_unstack = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Sort values
top_corr = corr_unstack.unstack().dropna().sort_values(ascending=False)

top_corr.head(10)

In [ ]:
pairs = [
    ('dtp3_coverage_pct', 'measles_coverage_pct'),
    ('u5mr', 'maternal_mortality_ratio'),
    ('sanitation_pct', 'safe_water_pct')
]

In [ ]:
for x, y in pairs:
    plt.figure()
    plt.scatter(df_clean[x], df_clean[y])
    plt.xlabel(x)
    plt.ylabel(y)
    plt.title(f"{x} vs {y}")
    plt.show()

The Pearson correlation matrix reveals strong relationships between variables. The highest correlations occur between vaccination coverage indicators, mortality variables, and infrastructure measures. Scatter plots confirm generally linear relationships, supporting the use of Pearson correlation, although some variables exhibit outlier influence and ceiling effects.

#### 1.3(b) Implications for Regression Modelling (VIF)

In [ ]:
pairs = [
    ('dtp3_coverage_pct', 'measles_coverage_pct'),
    ('sanitation_pct', 'safe_water_pct')
]

for x, y in pairs:
    spearman = df_clean[x].corr(df_clean[y], method='spearman')
    pearson = df_clean[x].corr(df_clean[y], method='pearson')

    print(f"{x} vs {y}:")
    print(f"  Pearson r = {pearson:.3f}")
    print(f"  Spearman ρ = {spearman:.3f}")
    print("------")

Spearman correlation was computed for strongly correlated predictors. The results are close to Pearson correlation, indicating strong monotonic relationships. High predictor correlation suggests potential multicollinearity, which may affect regression estimates and should be addressed in modelling.

#### 1.3(c) Grouped Comparison

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Order categories by median u5mr
order_income = df_clean.groupby('wb_income_group')['u5mr'].median().sort_values().index

plt.figure(figsize=(8,5))
sns.boxplot(x='wb_income_group', y='u5mr', data=df_clean, order=order_income)
plt.title("u5mr by Income Group")
plt.xticks(rotation=45)
plt.show()

In [ ]:
order_region = df_clean.groupby('wb_region')['u5mr'].median().sort_values().index

plt.figure(figsize=(10,5))
sns.boxplot(x='wb_region', y='u5mr', data=df_clean, order=order_region)
plt.title("u5mr by Region")
plt.xticks(rotation=45)
plt.show()

In [ ]:
summary_income = df_clean.groupby('wb_income_group')['u5mr'].agg(['mean', 'median', 'std', 'min', 'max'])
summary_income = summary_income.sort_values(by='median')

summary_region = df_clean.groupby('wb_region')['u5mr'].agg(['mean', 'median', 'std', 'min', 'max'])
summary_region = summary_region.sort_values(by='median')

summary_income.round(2), summary_region.round(2)

Boxplots were used to compare u5mr across income groups and regions. Clear differences in median and variability are observed.

Lower-income groups and regions such as Sub-Saharan Africa exhibit substantially higher mortality rates, while high-income groups show low and stable values. These results indicate that both income level and region are strong predictors of the outcome variable.

### 1.4 Outlier Analysis

In [ ]:
import numpy as np
from scipy import stats

variables = ['u5mr', 'sanitation_pct', 'safe_water_pct', 'dtp3_coverage_pct']

results = []

for col in variables:
    data = df_clean[col]

    # IQR method
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    iqr_outliers = data[(data < lower) | (data > upper)]

    # Z-score method
    z_scores = np.abs(stats.zscore(data))
    z_outliers = data[z_scores > 3]

    # Overlap
    overlap = set(iqr_outliers.index).intersection(set(z_outliers.index))

    results.append({
        'Variable': col,
        'IQR Outliers': len(iqr_outliers),
        'Z-score Outliers': len(z_outliers),
        'Overlap': len(overlap)
    })

import pandas as pd
pd.DataFrame(results)


Outliers were identified using both IQR and Z-score methods. The outcome variable u5mr shows a high number of outliers due to its skewed distribution. Percent-based variables show more IQR outliers but fewer extreme Z-score outliers. This indicates that the two methods capture different types of extreme values.

In [ ]:
df_clean.sort_values(by='u5mr', ascending=False).head(5)

The five most extreme values of u5mr were inspected and found to be plausible, as they correspond to low-income countries with known high mortality rates. These observations were retained to avoid bias. Transformation and winsorisation reduce skewness and variability, confirming their influence on the distribution.

In [ ]:
from scipy.stats.mstats import winsorize
import pandas as pd

# Winsorise top 1%
u5mr_wins = winsorize(df_clean['u5mr'], limits=[0, 0.01])

# BEFORE
mean_before = df_clean['u5mr'].mean()
sd_before = df_clean['u5mr'].std()
skew_before = df_clean['u5mr'].skew()

# AFTER
mean_after = u5mr_wins.mean()
sd_after = u5mr_wins.std()
skew_after = pd.Series(u5mr_wins).skew()

# Display
pd.DataFrame({
    'Before': [mean_before, sd_before, skew_before],
    'After': [mean_after, sd_after, skew_after]
}, index=['Mean', 'SD', 'Skewness']).round(2)

The five most extreme values of u5mr were examined and found to be plausible, as they correspond to low-income countries with historically high mortality rates. These observations were retained to avoid bias.

Winsorisation was applied as a robustness check. The results show a slight decrease in the mean (35.77 to 35.49), standard deviation (39.40 to 38.06), and skewness (2.04 to 1.73), indicating that extreme values contribute to distributional skewness and variability.

#### 1.4 (c)

In [ ]:
import statsmodels.api as sm
import numpy as np

# Define X and y
X = df_clean[['sanitation_pct']]
X = sm.add_constant(X)
y = df_clean['u5mr']

# Fit model
model = sm.OLS(y, X).fit()

# Cook's distance
influence = model.get_influence()
cooks_d = influence.cooks_distance[0]

# Threshold
n = len(df_clean)
threshold = 4 / n

print("Threshold:", threshold)

# Find influential points
influential = np.where(cooks_d > threshold)[0]
print("Number of influential points:", len(influential))


In [ ]:
df_no_outliers = df_clean.drop(index=influential)

# Refit model
X2 = df_no_outliers[['sanitation_pct']]
X2 = sm.add_constant(X2)
y2 = df_no_outliers['u5mr']

model2 = sm.OLS(y2, X2).fit()

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    'With Outliers': [
        model.params[1],
        model.rsquared,
        model.resid.std()
    ],
    'Without Outliers': [
        model2.params[1],
        model2.rsquared,
        model2.resid.std()
    ]
}, index=['Slope', 'R²', 'Residual SD'])

comparison.round(3)

Cook’s distance identified 448 influential observations. After removing these points, the model exhibits improved fit, with higher R² and lower residual variability. The slope remains largely unchanged, indicating that the relationship between sanitation and u5mr is stable and robust to influential observations.

In [ ]:
df_clean.to_csv('cleaned_dataset.csv', index=False)


In [ ]:
from google.colab import files
files.download('cleaned_dataset.csv')

## Part 2 - Inferential Statistics


---



### 2.1 Estimation and Confidence Interval

#### 2.1(a)

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# Function to compute both t-CI and bootstrap CI for a given group
def compute_ci(data, n_boot=10000, alpha=0.05):
    n = len(data)
    mean_val = np.mean(data)
    std_val = np.std(data, ddof=1)

    # t-distribution CI
    t_crit = stats.t.ppf(1 - alpha/2, df=n-1)
    margin_error = t_crit * (std_val / np.sqrt(n))
    ci_t = (mean_val - margin_error, mean_val + margin_error)

    # bootstrap CI
    rng = np.random.default_rng(42)
    boot_means = [np.mean(rng.choice(data, size=n, replace=True)) for _ in range(n_boot)]
    ci_boot = (np.percentile(boot_means, 2.5), np.percentile(boot_means, 97.5))

    return mean_val, ci_t, ci_boot

# Group by World Bank income group and compute CIs
results = []
for group, subset in df_clean.groupby("wb_income_group"):
    u5mr = subset['u5mr'].values
    mean_val, ci_t, ci_boot = compute_ci(u5mr)
    results.append({
        "Income Group": group,
        "Mean U5MR": mean_val,
        "t-CI Lower": ci_t[0],
        "t-CI Upper": ci_t[1],
        "t-CI Width": ci_t[1] - ci_t[0],
        "Bootstrap Lower": ci_boot[0],
        "Bootstrap Upper": ci_boot[1],
        "Bootstrap Width": ci_boot[1] - ci_boot[0],
        "Width Difference (Boot-t)": (ci_boot[1] - ci_boot[0]) - (ci_t[1] - ci_t[0])
    })

summary = pd.DataFrame(results)
print(summary)

In [ ]:
import matplotlib.pyplot as plt

summary.plot(x="Income Group", y=["t-CI Width", "Bootstrap Width"], kind="bar", figsize=(10,6))
plt.ylabel("CI Width")
plt.title("Comparison of CI Widths for Mean U5MR by Income Group")
plt.xticks(rotation=45)
plt.show()


The differences are small, but bootstrap tends to be more conservative (wider) in groups with skewed distributions, making it a safer choice when normality assumptions are questionable.

#### 2.1(b)

In [ ]:
# Assumption 1: Sample Size (CLT)

# Sample size per income group
n_table = (
    df_clean.groupby('wb_income_group')['u5mr']
    .count()
    .reset_index(name='Sample Size')
)

print(n_table)

All major income groups contained substantially more than 30 observations. Therefore, by the Central Limit Theorem, the sampling distribution of the sample mean is expected to be approximately normal even though the underlying U5MR distribution is skewed.

In [ ]:
# Assumption 2: Normality Assessment

# Shapiro-Wilk test
from scipy import stats

for group in df_clean['wb_income_group'].unique():

    x = df_clean.loc[
        df_clean['wb_income_group'] == group,
        'u5mr'
    ]

    stat, p = stats.shapiro(x)

    print(group)
    print("W =", round(stat,4))
    print("p =", p)
    print("----------------")

# Q-Q Plot
import scipy.stats as stats
import matplotlib.pyplot as plt

for group in df_clean['wb_income_group'].unique():

    x = df_clean.loc[
        df_clean['wb_income_group'] == group,
        'u5mr'
    ]

    plt.figure(figsize=(5,4))

    stats.probplot(x, dist="norm", plot=plt)

    plt.title(f"Q-Q Plot: {group}")

    plt.show()

Normality assessed using Shapiro-Wilk test. Since all p-values are substantially below 0.05, the null hypothesis of normality is rejected for every income group.

These findings are consistent with the earlier exploratory analysis, which showed that U5MR is positively skewed. Therefore, the normality assumption is formally violated.

In [ ]:
# Assumption 3: Independence

print(df_clean['country_name_wb'].nunique())
print(df_clean['year'].nunique())

The dataset consists of repeated country-year observations from 2000–2022. Consequently, observations from the same country across multiple years may be correlated, violating strict independence. This limitation cannot be formally tested using the available data. Therefore, the t-based confidence intervals may slightly underestimate the true uncertainty.

In [ ]:
# Skewness
for group in df_clean['wb_income_group'].unique():

    x = df_clean.loc[
        df_clean['wb_income_group'] == group,
        'u5mr'
    ]

    print(group)
    print("Skewness =", round(x.skew(),3))
    print()

Positive skewness confirms departure from normality.

Summary: Despite the violation of normality, the sample sizes for the four major income groups (n = 575 to n = 1,978) are sufficiently large for the Central Limit Theorem to apply. Furthermore, the bootstrap and t-based confidence intervals were nearly identical, with differences in interval width ranging from only 0.01 to 0.07 U5MR units. This empirical agreement suggests that the non-normality observed in the raw U5MR distributions has little practical impact on inference for the population mean.

#### 2.1(c)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# Outcome variable
u5mr = df_clean['u5mr'].values

# Bootstrap CI for median
rng = np.random.default_rng(42)
n_boot = 10000
boot_medians = [np.median(rng.choice(u5mr, size=len(u5mr), replace=True)) for _ in range(n_boot)]
ci_median = (np.percentile(boot_medians, 2.5), np.percentile(boot_medians, 97.5))

print("95% Bootstrap CI for Median U5MR:", ci_median)

# Spearman correlation
var1 = "gdp_per_capita_ppp"
var2 = "health_exp_per_capita_usd"
rho, _ = spearmanr(df_clean[var1], df_clean[var2])

# Bootstrap CI for Spearman correlation
boot_rhos = []
for _ in range(n_boot):
    sample_idx = rng.choice(len(df_clean), size=len(df_clean), replace=True)
    boot_rhos.append(spearmanr(df_clean[var1].iloc[sample_idx], df_clean[var2].iloc[sample_idx])[0])

ci_spearman = (np.percentile(boot_rhos, 2.5), np.percentile(boot_rhos, 97.5))

print(f"Spearman correlation between {var1} and {var2}: {rho:.3f}")
print("95% Bootstrap CI for Spearman correlation:", ci_spearman)

The bootstrap CI for the median collapsed to a single value (20.47, 20.47) because median imputation was applied to missing U5MR values. This imputation introduced many identical values equal to the median, making the resampled medians invariant. While this demonstrates the stability of the imputed dataset, it also masks true uncertainty. Therefore, the bootstrap CI for the median should be interpreted cautiously, as it reflects the imputation method rather than natural variability.

#### 2.1(d)

In [ ]:
import numpy as np
import pandas as pd

# Outcome variable
u5mr = df_clean['u5mr'].values
n_actual = len(u5mr)

# Required sample size derivation
z = 1.96
target_margin = 0.05  # in SD units
n_required = (z / target_margin) ** 2

print(f"Required sample size for ±0.05 SD margin at 95% CI: {int(np.ceil(n_required))}")
print(f"Actual sample size in dataset: {n_actual}")


The required sample size to achieve a margin of error of ±0.05 SD at 95% confidence is approximately 1537. Since our dataset contains 4991 observations, the actual precision is higher than required. Consequently, the confidence intervals for the mean U5MR are narrower than the design threshold, ensuring robust and reliable estimates.

### 2.2 Hypothesis Testing: One-Sample and Two-Sample Tests

#### 2.2(a)

1.   List item
2.   List item



In [ ]:
# One-Sample t-Test
from scipy import stats
t_stat, p_val_t = stats.ttest_1samp(df_clean['u5mr'], 50)
print(t_stat, p_val_t)

*   Result: t = −25.52, p < 0.001
*   Decision: Reject H₀ at α = 0.05. The mean U5MR is significantly lower than 50.

In [ ]:
# Welch's Two Sample t-Test
low = df_clean[df_clean['wb_income_group']=="Low income"]['u5mr']
high = df_clean[df_clean['wb_income_group']=="High income"]['u5mr']
w_t_stat, p_val_w = stats.ttest_ind(low, high, equal_var=False)
print(w_t_stat, p_val_w)


*   Result: t = 43.07, p < 0.001
*   Decision: Reject H₀ at α = 0.05. Low‑income countries have significantly higher mean U5MR.



In [ ]:
# Chi-Square Test of Independence
df_clean['dtp3_high'] = (df_clean['dtp3_coverage_pct'] >= 80).astype(int)
contingency = pd.crosstab(df_clean['wb_income_group'], df_clean['dtp3_high'])
chi2, p_val_c, dof, exp = stats.chi2_contingency(contingency)
print(chi2, p_val_c)

* Result: 𝜒2 = 1249.79, p < 0.001
* Decision: Reject H₀ at α = 0.05. Income group is strongly associated with vaccination coverage.

Summary: Together, these results demonstrate that U5MR is lower than the benchmark of 50, varies significantly by income group, and is linked to vaccination coverage, highlighting the interplay between economic status and child health outcomes.

#### 2.2(b)

In [ ]:
# Permutation test
import numpy as np

low = df_clean[df_clean['wb_income_group']=="Low income"]['u5mr'].values
high = df_clean[df_clean['wb_income_group']=="High income"]['u5mr'].values

obs_diff = np.mean(low) - np.mean(high)

combined = np.concatenate([low, high])
n_low = len(low)
rng = np.random.default_rng(42)

null_dist = []
for _ in range(10000):
    perm = rng.permutation(combined)
    null_diff = np.mean(perm[:n_low]) - np.mean(perm[n_low:])
    null_dist.append(null_diff)

# Empirical p-value
p_perm = np.mean(np.abs(null_dist) >= np.abs(obs_diff))
print("Observed difference:", obs_diff)
print("Permutation p-value:", p_perm)


In [ ]:
# Plot null distribution
import matplotlib.pyplot as plt

plt.hist(null_dist, bins=60, density=True, color="blue", edgecolor="white")
plt.axvline(obs_diff, color="red", lw=2, ls="--", label=f"Observed = {obs_diff:.2f}")
plt.xlabel("Permuted difference in means")
plt.ylabel("Density")
plt.title("Permutation Test Null Distribution")
plt.legend()
plt.show()


* The null distribution of mean differences was centered near zero, while the observed difference (89.47) lay far outside this range.
* The empirical p‑value was 0.0, consistent with the parametric p‑value of 3.71 × 10−183.
* Both approaches strongly reject H₀, confirming that under‑five mortality (u5mr) is significantly higher in low‑income countries compared to high‑income countries.

#### 2.2(c)

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

n = len(df_clean)

# 1. One-sample t-test effect size (Cohen's d)
cohen_d_one_sample = t_stat / np.sqrt(n)
print("Cohen's d (One-sample test):", cohen_d_one_sample)

# 2. Welch’s two-sample t-test effect size (Cohen's d)
low = df_clean[df_clean['wb_income_group']=="Low income"]['u5mr'].values
high = df_clean[df_clean['wb_income_group']=="High income"]['u5mr'].values

mean_diff = np.mean(low) - np.mean(high)
# pooled SD
s_pooled = np.sqrt(((len(low)-1)*np.var(low, ddof=1) + (len(high)-1)*np.var(high, ddof=1)) / (len(low)+len(high)-2))
cohen_d_welch = mean_diff / s_pooled
print("Cohen's d (Welch’s test):", cohen_d_welch)

# 3. Chi-square test effect size (Cramer’s V)
df_clean['dtp3_high'] = (df_clean['dtp3_coverage_pct'] >= 80).astype(int)
contingency = pd.crosstab(df_clean['wb_income_group'], df_clean['dtp3_high'])
chi2, p_val, dof, exp = stats.chi2_contingency(contingency)

n_total = contingency.sum().sum()
k = min(contingency.shape)
cramers_v = np.sqrt(chi2 / (n_total * (k-1)))
print("Cramer’s V (Chi-square test):", cramers_v)


*   The one‑sample test is statistically significant but has only a small effect size. With n = 4991, even negligible differences become significant.

*   The Welch’s test shows both statistical and practical importance, with an extremely large effect size.

*   The Chi‑square test shows a strong association between income group and vaccination coverage.

#### 2.2(d)

In [ ]:
import numpy as np

# Original p-values from your tests
p_values = np.array([
    p_val_t,  # One-sample t-test
    p_val_w,   # Welch’s t-test
    p_val_c   # Chi-square test
])

alpha = 0.05
m = len(p_values)
alpha_adj = alpha / m

# Apply Bonferroni correction
significant = p_values < alpha_adj

print("Adjusted alpha:", alpha_adj)
for i, p in enumerate(p_values, 1):
    print(f"Test {i}: p = {p:.2e}, Significant after Bonferroni? {significant[i-1]}")


In [ ]:
# p‑values vs adjusted alpha
import matplotlib.pyplot as plt

tests = ["One-sample t", "Welch’s t", "Chi-square"]
p_values = [p_val_t, p_val_w, p_val_c]
alpha_adj = 0.05 / 3

plt.bar(tests, p_values, color="pink")
plt.axhline(y=alpha_adj, color="purple", linestyle="--", label=f"Adjusted α = {alpha_adj:.4f}")
plt.yscale("log")  # log scale to show tiny p-values
plt.ylabel("p-value (log scale)")
plt.title("Bonferroni Correction: p-values vs Adjusted α")
plt.legend()
plt.show()


* Bonferroni reduces Type I error (false positives) but increases Type II error (false negatives).
* In this dataset, all effects are so strong that conclusions remain unchanged.


### 2.3 Tests for Categorical Variables

#### 2.3(a)

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

def chi_square_test(df, row_var, col_var):
    # Contingency table
    table = pd.crosstab(df[row_var], df[col_var])

    # Chi-square test
    chi2, p, dof, expected = chi2_contingency(table)

    # Cramer's V
    n = table.to_numpy().sum()
    r, c = table.shape
    cramers_v = np.sqrt(chi2 / (n * (min(r - 1, c - 1))))

    return table, chi2, dof, p, cramers_v

In [ ]:
# Pair 1: World Bank Income Group vs WHO Region
table1, chi2_1, dof_1, p_1, v_1 = chi_square_test(
    df_clean, 'wb_income_group', 'who_region'
)

print("Pair 1: World Bank Income Group vs WHO Region")
print("\nContingency Table:")
print(table1)

print("\nChi-square Test")
print("Chi-square =", round(chi2_1, 4))
print("df =", dof_1)
print("p-value =", p_1)
print("Cramer's V =", round(v_1, 4))

# Pair 2: World Bank Income Group vs U5MR group

# Create High/Low U5MR groups
median_u5mr = df_clean['u5mr'].median()

df_clean['u5mr_group'] = np.where(
    df_clean['u5mr'] > median_u5mr,
    'High U5MR',
    'Low U5MR'
)

# Contingency table
table2 = pd.crosstab(
    df_clean['wb_income_group'],
    df_clean['u5mr_group']
)

print("\n\nPair 2: World Bank Income Group vs U5MR group")
print("\nContingency Table:")
print(table2)

# Chi-square test
chi2, p, dof, _ = chi2_contingency(table2)

# Cramer's V
n = table2.to_numpy().sum()

cramers_v = np.sqrt(
    chi2 / (n * (min(table2.shape) - 1))
)

print("\nChi-square Test 2", round(chi2, 4))
print("df =", dof)
print("p-value =", p)
print("Cramer's V =", round(cramers_v, 4))

Both chi-square tests were statistically significant (p < 0.001), with a moderate association between income group and WHO region (Cramér's V = 0.390) and a strong association between income group and U5MR group (Cramér's V = 0.748), indicating that lower-income countries are disproportionately associated with higher under-five mortality rates.

#### 2.3(b)

In [ ]:
# Pair 1: wb_income_group vs who_region

table1 = pd.crosstab(
    df_clean['wb_income_group'],
    df_clean['who_region']
)

chi2_1, p_1, dof_1, expected1 = chi2_contingency(table1)

expected1_df = pd.DataFrame(
    expected1,
    index=table1.index,
    columns=table1.columns
)

problem_cells1 = expected1_df[expected1_df < 5].stack()

print("Pair 1: wb_income_group vs who_region")
print(f"Minimum Expected Count = {expected1.min():.2f}")
print(f"Cells with Expected Count < 5 = {(expected1 < 5).sum()}")

if len(problem_cells1) > 0:
    print("\nProblematic Cells:")
    print(problem_cells1)
else:
    print("\nNo problematic cells found.")


# Pair 2: wb_income_group vs u5mr_group

table2 = pd.crosstab(
    df_clean['wb_income_group'],
    df_clean['u5mr_group']
)

chi2_2, p_2, dof_2, expected2 = chi2_contingency(table2)

expected2_df = pd.DataFrame(
    expected2,
    index=table2.index,
    columns=table2.columns
)

problem_cells2 = expected2_df[expected2_df < 5].stack()

print("\n")
print("Pair 2: wb_income_group vs u5mr_group")
print(f"Minimum Expected Count = {expected2.min():.2f}")
print(f"Cells with Expected Count < 5 = {(expected2 < 5).sum()}")

if len(problem_cells2) > 0:
    print("\nProblematic Cells:")
    print(problem_cells2)
else:
    print("\nNo problematic cells found.")


# Conclusion

if (expected1 < 5).sum() == 0 and (expected2 < 5).sum() == 0:
    print("\nConclusion:")
    print("The minimum expected cell count condition is satisfied for both chi-square tests. No category collapsing or Fisher's exact test is required, and the chi-square results can be interpreted directly.")
else:
    print("\nConclusion:")
    print("Some expected counts are below 5. Categories should be collapsed where appropriate, or results should be interpreted with caution.")

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# Pair 1: wb_income_group vs who_region
# Problem: expected counts < 5 in the "Not classified" row
# Chosen solution: collapse "Not classified" into "Other"

df_test = df_clean.copy()

# Collapse the rare income category
df_test['wb_income_group_collapsed2'] = df_test['wb_income_group'].replace({
    'Not classified': 'Other'
})

# Reorder to keep a clean display
income_order = ['High income', 'Low income', 'Lower middle income', 'Upper middle income', 'Other']
df_test['wb_income_group_collapsed2'] = pd.Categorical(
    df_test['wb_income_group_collapsed2'],
    categories=income_order,
    ordered=False
)

table1 = pd.crosstab(df_test['wb_income_group_collapsed2'], df_test['who_region'])
chi2_1, p_1, dof_1, expected1 = chi2_contingency(table1)

expected1_df = pd.DataFrame(expected1, index=table1.index, columns=table1.columns)

print("Pair 1: wb_income_group (collapsed) vs who_region")
print("\nObserved Counts:")
print(table1)

print("\nExpected Counts:")
print(expected1_df.round(2))

print(f"\nMinimum Expected Count = {expected1.min():.2f}")
print(f"Cells with Expected Count < 5 = {(expected1 < 5).sum()}")

problem_cells1 = expected1_df[expected1_df < 5].stack()
if len(problem_cells1) > 0:
    print("\nProblematic Cells:")
    print(problem_cells1)
else:
    print("\nNo problematic cells found after collapsing.")

n1 = table1.to_numpy().sum()
cramers_v1 = np.sqrt(chi2_1 / (n1 * (min(table1.shape) - 1)))

print("\nChi-square Test Results:")
print(f"Chi-square = {chi2_1:.4f}")
print(f"df = {dof_1}")
print(f"p-value = {p_1:.6g}")
print(f"Cramer's V = {cramers_v1:.4f}")

# Pair 2: wb_income_group vs u5mr_group
# No issue with expected counts, keep as is

# Create u5mr_group if not already created
if 'u5mr_group' not in df_test.columns:
    median_u5mr = df_test['u5mr'].median()
    df_test['u5mr_group'] = np.where(df_test['u5mr'] > median_u5mr, 'High U5MR', 'Low U5MR')

table2 = pd.crosstab(df_test['wb_income_group'], df_test['u5mr_group'])
chi2_2, p_2, dof_2, expected2 = chi2_contingency(table2)

expected2_df = pd.DataFrame(expected2, index=table2.index, columns=table2.columns)

print("\n\nPair 2: wb_income_group vs u5mr_group")
print("\nObserved Counts:")
print(table2)

print("\nExpected Counts:")
print(expected2_df.round(2))

print(f"\nMinimum Expected Count = {expected2.min():.2f}")
print(f"Cells with Expected Count < 5 = {(expected2 < 5).sum()}")

problem_cells2 = expected2_df[expected2_df < 5].stack()
if len(problem_cells2) > 0:
    print("\nProblematic Cells:")
    print(problem_cells2)
else:
    print("\nNo problematic cells found.")

n2 = table2.to_numpy().sum()
cramers_v2 = np.sqrt(chi2_2 / (n2 * (min(table2.shape) - 1)))

print("\nChi-square Test Results:")
print(f"Chi-square = {chi2_2:.4f}")
print(f"df = {dof_2}")
print(f"p-value = {p_2:.6g}")
print(f"Cramer's V = {cramers_v2:.4f}")

In [ ]:
total_cells = expected1.size
cells_below_5 = (expected1 < 5).sum()

print("Total cells:", total_cells)
print("Cells < 5:", cells_below_5)
print("% cells < 5:", round(cells_below_5 / total_cells * 100, 1))
print("% cells >= 5:", round((total_cells - cells_below_5) / total_cells * 100, 1))

* Pair 1 has two expected counts below 5, but only 6.7% of cells violate the condition and no cell has an expected count below 1. Therefore, the chi-square test is retained and interpreted with caution.

* Pair 2 satisfies the minimum expected count assumption, so the chi-square test results can be interpreted directly.

#### 2.3(c)

In [ ]:
results = []

for region in sorted(df_clean['who_region'].unique()):

    subset = df_clean[df_clean['who_region'] == region]

    table = pd.crosstab(
        subset['wb_income_group'],
        subset['u5mr_group']
    )

    if table.shape[0] < 2 or table.shape[1] < 2:
        continue

    chi2, p, dof, _ = chi2_contingency(table)

    results.append({
        "WHO Region": region,
        "Chi-square": round(chi2, 2),
        "df": dof,
        "p-value": round(p, 5)
    })

summary = pd.DataFrame(results)

print(summary)

The association between World Bank income group and U5MR group remained statistically significant (p < 0.001) in all WHO regions, indicating that the aggregate relationship is not explained by regional confounding and there is no evidence of Simpson's paradox.

### 2.4 Distribution Fitting

#### 2.4(a)

In [ ]:
# Continuous Variable
x = df_clean['health_exp_per_capita_usd']

# Fit Normal Distribution (MLE)
mu_norm, sigma_norm = stats.norm.fit(x)

# Fit Lognormal Distribution (MLE)
shape_lognorm, loc_lognorm, scale_lognorm = stats.lognorm.fit(
    x,
    floc=0
)

# Histogram + Fitted PDFs
plt.figure(figsize=(10,6))

plt.hist(
    x,
    bins=30,
    density=True,
    alpha=0.6,
    label='Observed Data'
)

xmin, xmax = plt.xlim()
xx = np.linspace(xmin, xmax, 1000)

pdf_norm = stats.norm.pdf(
    xx,
    mu_norm,
    sigma_norm
)

pdf_lognorm = stats.lognorm.pdf(
    xx,
    shape_lognorm,
    loc_lognorm,
    scale_lognorm
)

plt.plot(xx, pdf_norm, linewidth=2,
         label='Normal Fit')

plt.plot(xx, pdf_lognorm, linewidth=2,
         label='Lognormal Fit')

plt.title('Health Expenditure per Capita: Normal vs Lognormal Fit')
plt.xlabel('Health Expenditure per Capita (USD)')
plt.ylabel('Density')
plt.legend()
plt.show()

# Q-Q Plot: Normal
plt.figure(figsize=(6,6))

stats.probplot(
    x,
    dist='norm',
    sparams=(mu_norm, sigma_norm),
    plot=plt
)

plt.title('Q-Q Plot: Normal Fit')
plt.show()

# Q-Q Plot: Lognormal
plt.figure(figsize=(6,6))

stats.probplot(
    np.log(x),
    dist='norm',
    plot=plt
)

plt.title('Q-Q Plot: Lognormal Fit')
plt.show()

# KS Tests
ks_norm = stats.kstest(
    x,
    'norm',
    args=(mu_norm, sigma_norm)
)

ks_lognorm = stats.kstest(
    x,
    'lognorm',
    args=(shape_lognorm,
          loc_lognorm,
          scale_lognorm)
)

print("\nNormal Distribution")
print("KS Statistic =", round(ks_norm.statistic,4))
print("p-value =", ks_norm.pvalue)

print("\nLognormal Distribution")
print("KS Statistic =", round(ks_lognorm.statistic,4))
print("p-value =", ks_lognorm.pvalue)

# Better Fit
if ks_lognorm.statistic < ks_norm.statistic:
    print("\nConclusion: Lognormal distribution provides the better fit.")
else:
    print("\nConclusion: Normal distribution provides the better fit.")

* Both KS tests rejected a perfect distributional fit (p < 0.001).
* The lognormal distribution (KS = 0.0848) fits health expenditure per capita substantially better than the normal distribution (KS = 0.2681).
* Consistent with the strong right-skew typically observed in economic indicators.

#### 2.4(b)

In [ ]:
for col in df_clean.columns:

    if pd.api.types.is_numeric_dtype(df_clean[col]):

        is_integer = (df_clean[col].dropna() % 1 == 0).all()

        print(f"{col}: Integer = {is_integer}")

In [ ]:
for col in ['population',
            'dtp3_coverage_pct',
            'measles_coverage_pct']:

    mean = df_clean[col].mean()
    var = df_clean[col].var()

    print(col)
    print("Mean:", round(mean,2))
    print("Variance:", round(var,2))
    print("Variance/Mean:", round(var/mean,2))
    print()

dtp3_coverage_pct is selected as it is an integer-valued variable, the overdispersion is clear but not absurd and it is directly related to vaccination coverage.

In [ ]:
# Count Variable: DTP3 Vaccination Coverage
x = df_clean['dtp3_coverage_pct'].astype(int)

# Poisson parameter
lambda_hat = x.mean()

# Dispersion ratio
mean_x = x.mean()
var_x = x.var(ddof=1)
dispersion_ratio = var_x / mean_x

print("Mean =", round(mean_x, 2))
print("Variance =", round(var_x, 2))
print("Dispersion Ratio =", round(dispersion_ratio, 2))

# Empirical Distribution
freq = x.value_counts(normalize=True).sort_index()

k = np.arange(x.min(), x.max() + 1)

plt.figure(figsize=(10,6))

plt.bar(
    freq.index,
    freq.values,
    alpha=0.6,
    label='Empirical Distribution'
)

# Poisson PMF
poisson_pmf = stats.poisson.pmf(k, lambda_hat)

plt.plot(
    k,
    poisson_pmf,
    linewidth=2,
    label=f'Poisson (λ={lambda_hat:.2f})'
)

# Negative Binomial Fit (if overdispersed)
if dispersion_ratio > 1.5:

    r = mean_x**2 / (var_x - mean_x)
    p = r / (r + mean_x)

    nb_pmf = stats.nbinom.pmf(k, r, p)

    plt.plot(
        k,
        nb_pmf,
        linewidth=2,
        label='Negative Binomial'
    )

    print("\nOverdispersion detected (ratio > 1.5)")
    print("Negative Binomial fitted.")

else:

    print("\nNo substantial overdispersion detected.")

plt.title('DTP3 Coverage: Empirical vs Fitted Distributions')
plt.xlabel('DTP3 Coverage (%)')
plt.ylabel('Probability')
plt.legend()
plt.show()


The Poisson model is not appropriate because the variance (198.08) substantially exceeds the mean (87.82), producing a dispersion ratio of 2.26. This indicates overdispersion, so a Negative Binomial distribution was fitted to better accommodate the excess variability in DTP3 vaccination coverage.

#### 2.4(c)

In [ ]:
# Check most skewed continuous variable

df = df_clean.copy()

# 1. Select numeric columns
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# 2. Compute skewness and count of non-missing values
skew_info = []
for col in num_cols:
    series = df[col].dropna()
    if series.size < 3:
        # skip columns with too few values to compute skewness reliably
        continue
    skewness = stats.skew(series, bias=False)   # Fisher-Pearson (unbiased)
    skew_info.append((col, skewness, series.size))

# 3. Create DataFrame and sort by absolute skewness
skew_df = pd.DataFrame(skew_info, columns=['variable', 'skewness', 'n']).set_index('variable')
skew_df['abs_skew'] = skew_df['skewness'].abs()
skew_df = skew_df.sort_values('abs_skew', ascending=False)

# 4. Show top variables
print("Top 10 numeric variables by absolute skewness:")
print(skew_df.head(10).round(3))

# 5. Pick the most skewed variable
most_skewed_var = skew_df.index[0]
most_skewed_skew = skew_df.iloc[0]['skewness']
print(f"\nMost skewed variable: {most_skewed_var} (skewness = {most_skewed_skew:.3f})")

# Quick plots for inspection
x = df[most_skewed_var]
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
sns.histplot(x, bins=40, kde=False)
plt.title(f"{most_skewed_var} histogram (skew={most_skewed_skew:.3f})")
plt.subplot(1,2,2)
sns.histplot(np.log1p(x[x>0]), bins=40, kde=False)
plt.title(f"log1p({most_skewed_var}) histogram")
plt.tight_layout()
plt.show()


In [ ]:
# CLT bootstrap demonstration for the most skewed variable: population

np.random.seed(2026)

x = df_clean['population'].values

# Quick diagnostics on original variable
orig_n = x.size
orig_mean = x.mean()
orig_var = x.var(ddof=1)
orig_skew = stats.skew(x, bias=False)
orig_kurt = stats.kurtosis(x, fisher=True, bias=False)

print("Original variable: population")
print(f"N = {orig_n}, mean = {orig_mean:.2f}, var = {orig_var:.2f}")
print(f"Skewness = {orig_skew:.3f}, Excess kurtosis = {orig_kurt:.3f}\n")

# Bootstrap settings
n_boot = 5000
sample_sizes = [5, 15, 30, 100]

means_by_n = {}
normality_results = {}

for n in sample_sizes:
    means = np.empty(n_boot)
    for i in range(n_boot):
        samp = np.random.choice(x, size=n, replace=True)
        means[i] = samp.mean()
    means_by_n[n] = means
    skew_means = stats.skew(means, bias=False)
    k2_stat, k2_p = stats.normaltest(means)   # D'Agostino K^2
    normality_results[n] = {
        "skewness": skew_means,
        "k2_stat": k2_stat,
        "k2_p": k2_p,
        "mean_of_means": means.mean(),
        "var_of_means": means.var(ddof=1)
    }

# Plot 2x2 panel
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for ax, n in zip(axes, sample_sizes):
    means = means_by_n[n]
    mu_emp = means.mean()
    sigma_emp = means.std(ddof=1)

    # Histogram of bootstrap means
    sns.histplot(means, bins=40, stat='density', color='C0', alpha=0.6, ax=ax, edgecolor=None)

    # Theoretical CLT normal curve using original mean and variance/n
    clt_mu = orig_mean
    clt_sigma = np.sqrt(orig_var / n)
    xx = np.linspace(means.min(), means.max(), 400)
    ax.plot(xx, stats.norm.pdf(xx, loc=clt_mu, scale=clt_sigma),
            color='k', linestyle='--', linewidth=2, label='CLT Normal (theoretical)')

    # Empirical normal fit to sample means
    ax.plot(xx, stats.norm.pdf(xx, loc=mu_emp, scale=sigma_emp),
            color='C3', linestyle='-', linewidth=2, label='Empirical Normal Fit')

    ax.set_title(f"n = {n} | mean = {mu_emp:.0f} | skew = {normality_results[n]['skewness']:.3f}")
    ax.set_xlabel('Sample mean (population)')
    ax.set_ylabel('Density')
    ax.legend()

plt.tight_layout()
plt.show()

# Print normality test summary
print("\nNormality test (D'Agostino K^2) and skewness for sample means")
for n in sample_sizes:
    res = normality_results[n]
    print(f"n = {n:3d} | skewness = {res['skewness']:.3f} | K2 = {res['k2_stat']:.2f} | p = {res['k2_p']:.3e}")


CLT check: for 'population' (skew=8.906) sample means remain non‑normal at n=5,15,30 and still depart at n=100 (D'Agostino p<<0.001); n>=30 is insufficient.


## Part 3 - Regression Modelling


---

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan

df = pd.read_csv('cleaned_dataset.csv')

# Recreate transformed columns
df['log_u5mr']       = np.log(df['u5mr'])
df['log_health_exp'] = np.log(df['health_exp_per_capita_usd'] + 1)
df['log_gdp']        = np.log(df['gdp_per_capita_ppp'] + 1)
df['log_maternal']   = np.log(df['maternal_mortality_ratio'] + 1)

print("Columns available:", df.columns.tolist())
print("Shape:", df.shape)

#### 3.1(a) — Fit SLR and report all stats

*   List item
*   List item



In [ ]:
# From Part 1.3, log_maternal has the strongest correlation with log_u5mr (r≈0.88)
# Both are log-transformed as per Part 1.2(c)

X_slr = sm.add_constant(df['log_maternal'].dropna())
y     = df['log_u5mr'].loc[X_slr.index]

model_slr = sm.OLS(y, X_slr).fit()
print(model_slr.summary())
print(f"\nRMSE: {np.sqrt(model_slr.mse_resid):.4f}")

#### 3.1(b) — JUSTIFY: Interpret β₁

In [ ]:
b0 = model_slr.params['const']
b1 = model_slr.params['log_maternal']

print("=== 3.1(b) Interpretation of β₁ ===\n")
print(f"β₀ (intercept) = {b0:.4f}")
print(f"β₁ (slope)     = {b1:.4f}")
print()
print("JUSTIFICATION:")
print(f"Since both outcome (log_u5mr) and predictor (log_maternal) are log-transformed,")
print(f"this is a log-log model. β₁ = {b1:.4f} is an elasticity:")
print(f"  → A 1% increase in maternal mortality ratio is associated with a {b1:.4f}%")
print(f"    increase in U5MR, holding all else constant.")
print()
print(f"The intercept β₀ = {b0:.4f} represents log(U5MR) when log(maternal_mortality) = 0,")
print(f"i.e., when maternal mortality ratio = 1. This is outside the realistic data range")
print(f"and is NOT directly interpretable in domain terms.")

####3.1(c) — Plot t-dist CI and Bootstrap CI for slope

In [ ]:
# t-distribution CI
ci_t = model_slr.conf_int(alpha=0.05)
lower_t = ci_t.loc['log_maternal', 0]
upper_t = ci_t.loc['log_maternal', 1]
print(f"t-dist CI:   [{lower_t:.4f}, {upper_t:.4f}]  width={upper_t - lower_t:.4f}")

# Bootstrap CI (10,000 resamples)
np.random.seed(42)
n = len(y)
boot_slopes = []
for _ in range(10000):
    idx = np.random.choice(n, n, replace=True)
    Xb  = sm.add_constant(df['log_maternal'].iloc[idx])
    yb  = df['log_u5mr'].iloc[idx]
    boot_slopes.append(sm.OLS(yb, Xb).fit().params['log_maternal'])

lower_b, upper_b = np.percentile(boot_slopes, [2.5, 97.5])
print(f"Bootstrap CI: [{lower_b:.4f}, {upper_b:.4f}]  width={upper_b - lower_b:.4f}")

# Plot both CIs
fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(['Bootstrap CI', 't-dist CI'],
        [upper_b - lower_b, upper_t - lower_t],
        left=[lower_b, lower_t],
        height=0.4, color=['steelblue', 'tomato'])
ax.axvline(model_slr.params['log_maternal'], color='black', lw=1.5, linestyle='--', label='Point estimate')
ax.set_xlabel('Slope (β₁)')
ax.set_title('95% CI for Slope: t-distribution vs Bootstrap')
ax.legend()
plt.tight_layout()
plt.show()

print("\nComparison: Both CIs are very similar in width.")
print("The bootstrap CI is slightly wider because it makes no normality assumption.")
print("When residuals are approximately normal (large n), both methods agree closely.")

#### 3.1(d) — Residual plots + JUSTIFY

1.   List item
2.   List item



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

fitted = model_slr.fittedvalues
resid  = model_slr.resid

# Residuals vs Fitted
axes[0].scatter(fitted, resid, alpha=0.3, s=10)
axes[0].axhline(0, color='red', lw=1)
axes[0].set_xlabel('Fitted values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted (SLR)')

# Q-Q Plot
stats.probplot(resid, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot of Residuals (SLR)')

plt.tight_layout()
plt.show()

print("=== 3.1(d) JUSTIFICATION ===\n")
print("Residuals vs Fitted:")
print("  - A random scatter around 0 indicates linearity and homoskedasticity (constant variance).")
print("  - If a fan shape appears, the E (Equal variance) assumption of LINE is violated.")
print("  - If a curve appears, the L (Linearity) assumption is violated.")
print()
print("Q-Q Plot:")
print("  - Points following the diagonal line indicate normally distributed residuals.")
print("  - Deviations at the tails indicate heavy-tailed or skewed residuals,")
print("    violating the N (Normality) assumption of LINE.")
print()
print("Consequence: Violated E (heteroskedasticity) inflates/deflates standard errors,")
print("making t-tests and CIs unreliable. Violated N affects small-sample inference,")
print("but with n > 4000, the CLT mitigates this for coefficient estimates.")

#### 3.2(a)+(b) — Four Nested Models + Comparison Table

In [ ]:
# Prepare dummies for TWO categorical predictors: wb_income_group + who_region
income_dummies = pd.get_dummies(df['wb_income_group'], prefix='income', drop_first=True).astype(int)
region_dummies = pd.get_dummies(df['who_region'],      prefix='region', drop_first=True).astype(int)

df_m = pd.concat([df, income_dummies, region_dummies], axis=1)
income_cols = [c for c in df_m.columns if c.startswith('income_')]
region_cols = [c for c in df_m.columns if c.startswith('region_')]

# Keep only rows with no NaN in any modelling column
all_cols = ['log_u5mr','log_maternal','log_health_exp','log_gdp',
            'dtp3_coverage_pct','sanitation_pct'] + income_cols + region_cols
df_m = df_m[all_cols].dropna().copy()
y_m  = df_m['log_u5mr']

def fit_ols(X_df, y_series):
    X = sm.add_constant(X_df)
    m = sm.OLS(y_series, X).fit()
    return m, np.sqrt(m.mse_resid)

# M1: SLR baseline
m1, r1 = fit_ols(df_m[['log_maternal']], y_m)

# M2: add log_health_exp and log_gdp
m2, r2 = fit_ols(df_m[['log_maternal','log_health_exp','log_gdp']], y_m)

# M3: add two categorical predictors (income group + WHO region)
X3_cols = ['log_maternal','log_health_exp','log_gdp','dtp3_coverage_pct'] + income_cols + region_cols
m3, r3  = fit_ols(df_m[X3_cols], y_m)

# M4: add interaction log_health_exp × income group (motivated by primary question)
for col in income_cols:
    df_m[f'hexp_x_{col}'] = df_m['log_health_exp'] * df_m[col]
interact_cols = [c for c in df_m.columns if c.startswith('hexp_x_')]
X4_cols = X3_cols + interact_cols
m4, r4  = fit_ols(df_m[X4_cols], y_m)

# Comparison table
comp = pd.DataFrame({
    'Model':   ['M1: SLR', 'M2: +health+gdp', 'M3: +income+region', 'M4: +interaction'],
    'Adj_R2':  [m1.rsquared_adj, m2.rsquared_adj, m3.rsquared_adj, m4.rsquared_adj],
    'RMSE':    [r1, r2, r3, r4],
    'AIC':     [m1.aic, m2.aic, m3.aic, m4.aic],
    'BIC':     [m1.bic, m2.bic, m3.bic, m4.bic],
})
print(comp.round(4).to_string(index=False))

print("\n=== JUSTIFICATION ===")
print("Adjusted R² penalises each added predictor by degrees of freedom consumed.")
print("If a new predictor explains less variance than the penalty costs, Adj-R² drops.")
print("AIC and BIC may disagree because BIC's penalty is log(n) per parameter vs 2 for AIC.")
print(f"With n={len(y_m)}, log(n)={np.log(len(y_m)):.2f}, so BIC penalises complexity more heavily,")
print("favouring simpler models than AIC does.")

#### 3.2(c) — Preferred Model (M4) full summary + interaction visualisation

In [ ]:
# We use M4 as preferred (it directly addresses the primary question:
# does health expenditure effect vary by income group?)
print(m4.summary())

print("\n=== Coefficient Interpretation (M4) ===")
print("Reference categories: Low income (income), AFR - African Region (region)")
print()
print("log_maternal:    A 1% rise in maternal mortality → β% rise in U5MR (elasticity)")
print("log_health_exp:  Effect of health expenditure FOR the reference group (Low income)")
print("log_gdp:         A 1% rise in GDP per capita → β% change in U5MR")
print("dtp3_coverage:   Each 1pp rise in DTP3 vaccination → β% change in U5MR")
print("income_X:        Difference in log(U5MR) between income group X and Low income,")
print("                 when health expenditure = 0 (i.e. at the intercept)")
print("region_X:        Difference in log(U5MR) between region X and AFR")
print("hexp_x_income_X: How the slope of log_health_exp CHANGES for income group X")
print("                 relative to Low income countries")

In [ ]:
# Visualisation: regression lines of log_health_exp vs log_u5mr per income group
fig, ax = plt.subplots(figsize=(9, 5))

colors = plt.cm.tab10.colors
groups = df['wb_income_group'].dropna().unique()

for i, grp in enumerate(sorted(groups)):
    sub = df[df['wb_income_group'] == grp].dropna(subset=['log_health_exp','log_u5mr'])
    ax.scatter(sub['log_health_exp'], sub['log_u5mr'],
               alpha=0.2, s=8, color=colors[i])
    if len(sub) > 5:
        coef = np.polyfit(sub['log_health_exp'], sub['log_u5mr'], 1)
        xr   = np.linspace(sub['log_health_exp'].min(), sub['log_health_exp'].max(), 50)
        ax.plot(xr, np.polyval(coef, xr), lw=2, color=colors[i], label=grp)

ax.set_xlabel('log(Health Expenditure per Capita)')
ax.set_ylabel('log(U5MR)')
ax.set_title('Interaction: Effect of Health Expenditure on U5MR varies by Income Group')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

#### 3.2(d) — VIF + JUSTIFY

---



In [ ]:
X4_const = sm.add_constant(df_m[X4_cols])
vif_df = pd.DataFrame({
    'Variable': X4_const.columns,
    'VIF': [variance_inflation_factor(X4_const.values, i)
            for i in range(X4_const.shape[1])]
}).sort_values('VIF', ascending=False)
print(vif_df.to_string(index=False))

high_vif = vif_df[vif_df['VIF'] > 10]
print(f"\nPredictors with VIF > 10: {len(high_vif)}")
if len(high_vif) > 0:
    print(high_vif.to_string(index=False))

print("\n=== JUSTIFICATION ===")
print("Multicollinearity inflates SE of correlated coefficients by factor √VIF.")
print("Example: VIF=9 → SE inflated by √9=3 → t-statistic shrinks by factor 3.")
print("This makes genuinely predictive variables appear non-significant (Type II error),")
print("even when overall model F-test is significant — because the F-test only asks")
print("whether predictors JOINTLY explain variance, not individually.")
print("Interaction terms (hexp_x_income) naturally have high VIF with their components.")
print("This is expected and acceptable — we center log_health_exp to reduce it if needed.")

#### 3.2(e) — Full 4-panel diagnostic suite

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

fitted4    = m4.fittedvalues
resid4     = m4.resid
influence4 = m4.get_influence()
leverage4  = influence4.hat_matrix_diag
std_resid4 = influence4.resid_studentized_internal

# (i) Residuals vs Fitted — check linearity + homoskedasticity
axes[0,0].scatter(fitted4, resid4, alpha=0.3, s=8)
axes[0,0].axhline(0, color='red', lw=1)
axes[0,0].set_xlabel('Fitted values')
axes[0,0].set_ylabel('Residuals')
axes[0,0].set_title('(i) Residuals vs Fitted\n[Look for: random scatter around 0]')

# (ii) Q-Q plot — check normality of residuals
stats.probplot(resid4, dist='norm', plot=axes[0,1])
axes[0,1].set_title('(ii) Q-Q Plot\n[Look for: points on diagonal line]')

# (iii) Scale-Location — check homoskedasticity (flat line = good)
axes[1,0].scatter(fitted4, np.sqrt(np.abs(std_resid4)), alpha=0.3, s=8)
axes[1,0].set_xlabel('Fitted values')
axes[1,0].set_ylabel('√|Standardised Residuals|')
axes[1,0].set_title('(iii) Scale-Location\n[Look for: flat horizontal trend]')

# (iv) Residuals vs Leverage — identify influential points
axes[1,1].scatter(leverage4, std_resid4, alpha=0.3, s=8)
axes[1,1].axhline(0, color='red', lw=1)
axes[1,1].axhline(3,  color='orange', lw=1, linestyle='--', label='|std resid|=3')
axes[1,1].axhline(-3, color='orange', lw=1, linestyle='--')
axes[1,1].set_xlabel('Leverage')
axes[1,1].set_ylabel('Standardised Residuals')
axes[1,1].set_title("(iv) Residuals vs Leverage\n[Look for: points beyond Cook's D contours]")
axes[1,1].legend(fontsize=7)

plt.tight_layout()
plt.show()

#### 3.2(f) — Breusch-Pagan test + HC3 robust SEs + JUSTIFY

In [ ]:
bp_stat, bp_p, _, _ = het_breuschpagan(m4.resid, m4.model.exog)
print(f"Breusch-Pagan test: stat={bp_stat:.3f},  p={bp_p:.4f}")
print("→ If p < 0.05: heteroskedasticity is present, use HC3 robust SEs.\n")

m4_hc3 = m4.get_robustcov_results(cov_type='HC3')

se_compare = pd.DataFrame({
    'OLS SE':        m4.bse,
    'HC3 SE':        m4_hc3.bse,
    'Ratio HC3/OLS': (m4_hc3.bse / m4.bse).round(3)
}).round(4)
print(se_compare.to_string())

print("\n=== JUSTIFICATION ===")
print("Heteroskedasticity means residual variance is not constant across fitted values.")
print("OLS standard errors assume constant variance (homoskedasticity).")
print("When violated, OLS SEs are biased — usually underestimated for high-leverage points.")
print("HC3 (heteroskedasticity-consistent) SEs correct for this without changing coefficients.")
print("Predictors where HC3 SE >> OLS SE should be interpreted more cautiously.")

#### 3.3(a) — Polynomial term for non-linearity

In [3]:
# dtp3_coverage_pct likely has diminishing returns (non-linear) with log_u5mr
plt.figure(figsize=(6,4))
plt.scatter(df_m['dtp3_coverage_pct'], y_m, alpha=0.2, s=8)
plt.xlabel('DTP3 Coverage (%)')
plt.ylabel('log(U5MR)')
plt.title('DTP3 Coverage vs log(U5MR) — checking non-linearity')
plt.tight_layout()
plt.show()

# Add quadratic term
df_m['dtp3_sq'] = df_m['dtp3_coverage_pct'] ** 2

X4_poly_cols = ['log_maternal','log_health_exp','log_gdp',
                'dtp3_coverage_pct','dtp3_sq'] + income_cols + region_cols + interact_cols
m4_poly, _ = fit_ols(df_m[X4_poly_cols], y_m)

print(f"M4 (linear DTP3)  — AIC: {m4.aic:.2f},  Adj-R²: {m4.rsquared_adj:.4f}")
print(f"M4 (poly DTP3)    — AIC: {m4_poly.aic:.2f},  Adj-R²: {m4_poly.rsquared_adj:.4f}")
print()
print("If poly AIC is lower → quadratic term is warranted (reduces model misfit more than")
print("it costs in complexity). If AIC barely changes, linear term is sufficient.")

NameError: name 'df_m' is not defined

<Figure size 600x400 with 0 Axes>

#### 3.3(b) — Engineered feature: Vaccination × Sanitation Index

In [ ]:
# Domain rationale:
# Child survival depends on BOTH vaccination AND safe sanitation together.
# A vaccinated child in a low-sanitation environment still faces high mortality risk
# from diarrhoeal disease and co-infections. Their combined effect is multiplicative.

df_m['vacc_sanit'] = (df_m['dtp3_coverage_pct'] / 100) * (df_m['sanitation_pct'] / 100)

X4_eng_cols = ['log_maternal','log_health_exp','log_gdp',
               'dtp3_coverage_pct','vacc_sanit'] + income_cols + region_cols + interact_cols
m4_eng, _ = fit_ols(df_m[X4_eng_cols], y_m)

print(f"M4 without vacc_sanit — AIC: {m4.aic:.2f},  Adj-R²: {m4.rsquared_adj:.4f}")
print(f"M4 with vacc_sanit    — AIC: {m4_eng.aic:.2f},  Adj-R²: {m4_eng.rsquared_adj:.4f}")
print()
print("If AIC decreases: the feature adds genuine predictive value beyond its components.")

#### 3.3(c) — Modelling Choices

---



In [ ]:
print("""
=== 3.3(c) MODELLING CHOICES ===

Variable Selection:
log(maternal_mortality_ratio) was chosen as the primary predictor because it showed
the strongest correlation with log(U5MR) in Part 1.3 (r ≈ 0.88). This is also
theoretically justified — high maternal mortality and high child mortality share the
same root causes: weak health systems, poor nutrition, and limited access to care.
A reasonable analyst might instead start with log(health_expenditure), which is more
directly linked to the primary question about health system inputs, but would sacrifice
some model R² at the baseline stage.

Transformations:
Log transformations were applied to all right-skewed continuous predictors (health
expenditure, GDP, maternal mortality) to linearise multiplicative relationships and
reduce the influence of extreme outliers (e.g. Qatar vs Somalia in GDP). The outcome
log(U5MR) was established in Part 1.2(c). An alternative is square-root transformation,
which is more conservative but retains more variation at the lower end of the scale.

Interaction Term:
log_health_exp × wb_income_group is motivated directly by the primary question, which
asks whether the effect of health expenditure is modified by income group. Without this
interaction, the model assumes the same return on health spending for Low income and
High income countries — an implausible assumption. A reasonable alternative is full
stratification (separate models per income group), but this prevents a formal
interaction significance test and loses statistical power.

Outlier Treatment:
Influential observations (Cook's D > 4/n) were retained. Removing them would discard
legitimate data from countries with genuine extreme conditions (e.g. Mali, Chad),
biasing estimates toward middle-income countries and undermining generalisability.
""")

#### 3.4(a) — CIs for all coefficients + F-test vs t-test

In [ ]:
ci_m4 = m4.conf_int(alpha=0.05)
ci_m4.columns = ['CI_lower', 'CI_upper']
ci_m4['coef']          = m4.params
ci_m4['p_value']       = m4.pvalues.round(4)
ci_m4['excludes_zero'] = (ci_m4['CI_lower'] > 0) | (ci_m4['CI_upper'] < 0)
print(ci_m4.round(4).to_string())

print(f"\nF-statistic: {m4.fvalue:.2f},  p-value: {m4.f_pvalue:.4e}")

print("\n=== JUSTIFICATION ===")
print("Each t-test asks: does THIS predictor add explanatory power given all others?")
print("The F-test asks: do ALL predictors jointly explain variance beyond the intercept?")
print()
print("The F-test can be significant while ALL individual t-tests are not when predictors")
print("are severely multicollinear. Each predictor appears redundant given the others,")
print("so no single t-statistic is large enough — yet together they explain substantial")
print("variance. This is a hallmark of multicollinearity, not model failure.")

#### 3.4(b) — Prediction Interval vs Confidence Interval

In [ ]:
# Hypothetical: Lower-middle income, health_exp=200 USD, GDP=3000, maternal_mort=30, DTP3=75%
new_obs = {'log_maternal':      np.log(30),
           'log_health_exp':    np.log(200),
           'log_gdp':           np.log(3000),
           'dtp3_coverage_pct': 75.0}

# Set all dummies to 0 (reference = Low income, AFR region)
for col in income_cols + region_cols + interact_cols:
    new_obs[col] = 0.0

new_df = pd.DataFrame([new_obs])[X4_cols]
new_X  = sm.add_constant(new_df, has_constant='add')

pred = m4.get_prediction(new_X)
ps   = pred.summary_frame(alpha=0.05)

print("Hypothetical country: Lower-middle income, health_exp=$200, GDP=$3000,")
print("                      maternal_mortality=30, DTP3 coverage=75%\n")
print(ps[['mean','mean_ci_lower','mean_ci_upper','obs_ci_lower','obs_ci_upper']].round(4))
print()
print(f"Predicted U5MR:           {np.exp(ps['mean'].values[0]):.1f} per 1,000 live births")
print(f"95% CI for mean response: ({np.exp(ps['mean_ci_lower'].values[0]):.1f}, {np.exp(ps['mean_ci_upper'].values[0]):.1f})")
print(f"95% Prediction Interval:  ({np.exp(ps['obs_ci_lower'].values[0]):.1f}, {np.exp(ps['obs_ci_upper'].values[0]):.1f})")

print("\n=== JUSTIFICATION ===")
print("The PI is always wider than the CI for the mean because:")
print("  CI captures only uncertainty about WHERE the true mean lies (shrinks with n).")
print("  PI also captures individual-level variability (σ², the irreducible noise).")
print("  Even with infinite data, a single new country would still vary around the mean.")
print("  Mathematically: PI variance = CI variance + σ²  →  PI is always wider.")

#### 3.4(c) — Partial R² for log_health_exp

In [ ]:
# Partial R² = unique variance explained by log_health_exp
# = (RSS without it - RSS with it) / RSS without it

X4_reduced_cols = [c for c in X4_cols if c != 'log_health_exp' and not c.startswith('hexp_x_')]
m4_reduced, _ = fit_ols(df_m[X4_reduced_cols], y_m)

rss_full    = np.sum(m4.resid ** 2)
rss_reduced = np.sum(m4_reduced.resid ** 2)
partial_r2  = (rss_reduced - rss_full) / rss_reduced

print(f"R² (full model M4):     {m4.rsquared:.4f}")
print(f"R² (without health exp): {m4_reduced.rsquared:.4f}")
print(f"Partial R² for log_health_exp: {partial_r2:.4f}")
print()
print(f"Interpretation: log(health expenditure) — including its interaction with income group —")
print(f"uniquely explains {partial_r2*100:.1f}% of the variance in log(U5MR)")
print(f"that cannot be accounted for by maternal mortality, GDP, DTP3, and region alone.")
print(f"This is the variable most central to the primary research question.")

## Part 4 - Classification and Logistic Regression

---

### 4.1 Binary Outcome Definition and Baseline

#### 4.1(a)

#### 4.1(b)

### 4.2 Logistic Regression Modelling

#### 4.2(a)

#### 4.2(b)

#### 4.2(c)

#### 4.2(d)

### 4.3 Model Evaluation

#### 4.3(a)

#### 4.3(b)

#### 4.3(c)

#### 4.3(d)

# Model Validation & Regularization
Cross-validation, Ridge/Lasso, bias-variance